In [1]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from tqdm import tqdm

from Utils.DataUtils import build_ae_dataloaders

from Models.FraudModel import FraudMLP
from Models.AutoEncoder import AutoEncoder

from Utils.TrainUtils import get_device, train_model

In [2]:
train_loader, val_loader, test_loader, input_dim = build_ae_dataloaders(batch_size=256)

[INFO] Project root: c:\Users\mengt\Documents\DeepLearning\DL_project
[INFO] Data dir: c:\Users\mengt\Documents\DeepLearning\DL_project\data
[INFO] Loading train data from: c:\Users\mengt\Documents\DeepLearning\DL_project\data\train_transaction.csv
[INFO] Loading test data from: c:\Users\mengt\Documents\DeepLearning\DL_project\data\test_transaction.csv
[INFO] Train shape: (590540, 394)
[INFO] Test shape: (506691, 393)


c:\Users\mengt\Documents\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
c:\Users\mengt\Documents\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
c:\Users\mengt\Documents\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which

[INFO] Train: (472432, 776)
[INFO] Val: (59054, 776)
[INFO] Test: (59054, 776)


In [35]:
DEVICE = get_device()
print(DEVICE)

# autoencoder = AutoEncoder (
#     input_dim=input_dim,
#     latent_dim=16,              
#     hidden_dims=[128, 64]
# ).to(DEVICE)

# autoencoder.load_state_dict(torch.load("checkpoints/ae_best_L16.pt", map_location=DEVICE))

# Trying Seline's autoencoder i may have made my latent state too small
autoencoder = AutoEncoder(
    input_dim=input_dim,
    latent_dim=256,              
    hidden_dims=[128, 64, 32],
    noise_std=0.01,
).to(DEVICE)

autoencoder.load_state_dict(torch.load("checkpoints/autoencoder.pt", map_location=DEVICE)["model_state_dict"])

# model = FraudMLP (
#     input_dim=input_dim,
#     hidden_dims=[1024, 512, 256, 128, 64],
#     gated=True,
#     dropout=0.3,
#     encoder=autoencoder,
#     freeze_encoder=True
# ).to(DEVICE)

# Train baseline
model = FraudMLP (
    input_dim=input_dim,
    hidden_dims=[1024, 512, 256, 128, 64],
    gated=True,
    dropout=0.1
).to(DEVICE)

cuda


In [36]:
y = train_loader.dataset.y
pos_weight = (y == 0).sum().item() / (y == 1).sum().item()

print("Pos Weight:", pos_weight)

Pos Weight: 27.580278281911674


In [37]:
model = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=120,
    lr=5e-5,
    device=DEVICE,
    pos_weight=pos_weight*0.5,
    save_path="GatedMLP_V6",
    optimize_for="f2",
    num_thresholds=300,
)

[INFO] Saved best model

Epoch 1/120
Train Loss:      0.637256
Val Loss:        0.576500
Val @ 0.5 -> Acc: 0.9479 | Prec: 0.3444 | Rec: 0.5414 | F1: 0.4210 | F2: 0.4858
Best threshold:  0.4504
Best metrics -> Acc: 0.9382 | Prec: 0.3038 | Rec: 0.5936 | F1: 0.4019 | F2: 0.4985
Confusion @ best -> TP: 1227 | FP: 2812 | TN: 54175 | FN: 840
Epoch time:      12.77s
[INFO] Saved best model

Epoch 2/120
Train Loss:      0.577880
Val Loss:        0.534412
Val @ 0.5 -> Acc: 0.9359 | Prec: 0.3013 | Rec: 0.6299 | F1: 0.4076 | F2: 0.5171
Best threshold:  0.5819
Best metrics -> Acc: 0.9526 | Prec: 0.3830 | Rec: 0.5815 | F1: 0.4619 | F2: 0.5269
Confusion @ best -> TP: 1202 | FP: 1936 | TN: 55051 | FN: 865
Epoch time:      11.60s
[INFO] Saved best model

Epoch 3/120
Train Loss:      0.544119
Val Loss:        0.520902
Val @ 0.5 -> Acc: 0.9311 | Prec: 0.2912 | Rec: 0.6754 | F1: 0.4069 | F2: 0.5344
Best threshold:  0.5805
Best metrics -> Acc: 0.9473 | Prec: 0.3582 | Rec: 0.6386 | F1: 0.4590 | F2: 0.5522


In [38]:
# TESTING
import json

from Utils.TrainUtils import evaluate

DEVICE = get_device()

In [39]:
with open("checkpoints/GatedMLP_V6/summary.json", "r", encoding="utf-8") as f:
    summary_v2 = json.load(f)

threshold_v2 = summary_v2["best_val_metrics"]["best_threshold"]

model_v2 = FraudMLP(
    input_dim=input_dim,
    hidden_dims=[1024, 512, 256, 128, 64],
    gated=True,
    dropout=0.1,
    use_norm=True,
    encoder=None,
    freeze_encoder=False,
).to(DEVICE)

ckpt_v2 = torch.load("checkpoints/GatedMLP_V6/best.pt", map_location=DEVICE)
if "model_state_dict" in ckpt_v2:
    model_v2.load_state_dict(ckpt_v2["model_state_dict"])
else:
    model_v2.load_state_dict(ckpt_v2)

model_v2.eval()

criterion_v2 = torch.nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(summary_v2["pos_weight"], dtype=torch.float32, device=DEVICE)
)

test_metrics_v2 = evaluate(
    model=model_v2,
    loader=test_loader,
    criterion=criterion_v2,
    device=DEVICE,
    threshold=threshold_v2,
    sweep_thresholds=False,
)

print("=== GatedMLP_V6 Test ===")
for k in ["loss", "threshold", "accuracy", "precision", "recall", "f1", "f2", "tp", "fp", "tn", "fn"]:
    print(f"{k}: {test_metrics_v2[k]}")

test_metrics_v2_fixed = evaluate(
    model=model_v2,
    loader=test_loader,
    criterion=criterion_v2,
    device=DEVICE,
    threshold=0.5,
    sweep_thresholds=False,
)

print("=== GatedMLP_V6 Test (0.5 Threshold) ===")
for k in ["loss", "threshold", "accuracy", "precision", "recall", "f1", "f2", "tp", "fp", "tn", "fn"]:
    print(f"{k}: {test_metrics_v2_fixed[k]}")

=== GatedMLP_V6 Test ===
loss: 0.8150083881387592
threshold: 0.7119498252868652
accuracy: 0.9737020354250392
precision: 0.611667392248099
recall: 0.6800580832493706
f1: 0.6440522526319669
f2: 0.6651832191553039
tp: 1405
fp: 892
tn: 56096
fn: 661
=== GatedMLP_V6 Test (0.5 Threshold) ===
loss: 0.8150083881387592
threshold: 0.5
accuracy: 0.9636434449823952
precision: 0.4865044984988787
recall: 0.7066795740527266
f1: 0.5762778716234734
f2: 0.6480248533021254
tp: 1460
fp: 1541
tn: 55447
fn: 606


In [40]:
with open("checkpoints/GatedMLP_AE16_V4/summary.json", "r", encoding="utf-8") as f:
    summary_ae16 = json.load(f)

threshold_ae16 = summary_ae16["best_val_metrics"]["best_threshold"]

autoencoder = AutoEncoder(
    input_dim=input_dim,
    latent_dim=16,
    hidden_dims=[128, 64],
    noise_std=0.01,
).to(DEVICE)

ae_ckpt = torch.load("checkpoints/ae_best_L16.pt", map_location=DEVICE)
if "model_state_dict" in ae_ckpt:
    autoencoder.load_state_dict(ae_ckpt["model_state_dict"])
else:
    autoencoder.load_state_dict(ae_ckpt)
autoencoder.eval()

model_ae16 = FraudMLP(
    input_dim=input_dim,
    hidden_dims=[1024, 512, 256, 128, 64],
    gated=True,
    dropout=0.1,
    use_norm=True,
    encoder=autoencoder,
    freeze_encoder=True,
).to(DEVICE)

ckpt_ae16 = torch.load("checkpoints/GatedMLP_AE16_V4/best.pt", map_location=DEVICE)
if "model_state_dict" in ckpt_ae16:
    model_ae16.load_state_dict(ckpt_ae16["model_state_dict"])
else:
    model_ae16.load_state_dict(ckpt_ae16)

model_ae16.eval()

criterion_ae16 = torch.nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(summary_ae16["pos_weight"], dtype=torch.float32, device=DEVICE)
)

test_metrics_ae16 = evaluate(
    model=model_ae16,
    loader=test_loader,
    criterion=criterion_ae16,
    device=DEVICE,
    threshold=threshold_ae16,
    sweep_thresholds=False,
)

print("=== GatedMLP_AE16 Test ===")
for k in ["loss", "threshold", "accuracy", "precision", "recall", "f1", "f2", "tp", "fp", "tn", "fn"]:
    print(f"{k}: {test_metrics_ae16[k]}")

test_metrics_ae16_fixed = evaluate(
    model=model_ae16,
    loader=test_loader,
    criterion=criterion_ae16,
    device=DEVICE,
    threshold=0.5,
    sweep_thresholds=False,
)

print("=== GatedMLP_AE16 Test (0.5 threshold) ===")
for k in ["loss", "threshold", "accuracy", "precision", "recall", "f1", "f2", "tp", "fp", "tn", "fn"]:
    print(f"{k}: {test_metrics_ae16_fixed[k]}")

=== GatedMLP_AE16 Test ===
loss: 0.6393033988064244
threshold: 0.5784
accuracy: 0.9744301825446245
precision: 0.6213973799099503
recall: 0.6887705711486507
f1: 0.6533516938164663
f2: 0.6741519781648974
tp: 1423
fp: 867
tn: 56121
fn: 643
=== GatedMLP_AE16 Test (0.5 threshold) ===
loss: 0.6393033988064244
threshold: 0.5
accuracy: 0.970112100788944
precision: 0.557377049178203
recall: 0.7076476282637578
f1: 0.6235871138565484
f2: 0.6714430031301926
tp: 1462
fp: 1161
tn: 55827
fn: 604


In [41]:
# AE256 from selina

with open("checkpoints/GatedMLP_AE256_V1/summary.json", "r", encoding="utf-8") as f:
    summary_ae256 = json.load(f)

threshold_ae256 = summary_ae256["best_val_metrics"]["best_threshold"]

# Rebuild the autoencoder first
autoencoder = AutoEncoder(
    input_dim=input_dim,
    latent_dim=256,
    hidden_dims=[128, 64, 32],
    noise_std=0.01,
).to(DEVICE)

ae_ckpt = torch.load("checkpoints/autoencoder.pt", map_location=DEVICE)
ae_state_dict = ae_ckpt["model_state_dict"] if "model_state_dict" in ae_ckpt else ae_ckpt
autoencoder.load_state_dict(ae_state_dict)
autoencoder.eval()

# Now build the FraudMLP with the encoder attached
model_ae256 = FraudMLP(
    input_dim=input_dim,
    hidden_dims=[1024, 512, 256, 128, 64],
    gated=True,
    dropout=0.1,
    encoder=autoencoder,
    freeze_encoder=True,
).to(DEVICE)

# Load the trained FraudMLP checkpoint
ckpt_ae256 = torch.load("checkpoints/GatedMLP_AE256_V1/best.pt", map_location=DEVICE)
state_dict_ae256 = ckpt_ae256["model_state_dict"] if "model_state_dict" in ckpt_ae256 else ckpt_ae256

model_ae256.load_state_dict(state_dict_ae256)
model_ae256.eval()

criterion_ae256 = torch.nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(summary_ae256["pos_weight"], dtype=torch.float32, device=DEVICE)
)

test_metrics_ae256 = evaluate(
    model=model_ae256,
    loader=test_loader,
    criterion=criterion_ae256,
    device=DEVICE,
    threshold=threshold_ae256,
    sweep_thresholds=False,
)

print("=== GatedMLP_AE256 Test ===")
for k in ["loss", "threshold", "accuracy", "precision", "recall", "f1", "f2", "tp", "fp", "tn", "fn"]:
    print(f"{k}: {test_metrics_ae256[k]}")

test_metrics_ae256_fixed = evaluate(
    model=model_ae256,
    loader=test_loader,
    criterion=criterion_ae256,
    device=DEVICE,
    threshold=0.5,
    sweep_thresholds=False,
)

print("=== GatedMLP_AE256 Test (0.5 threshold) ===")
for k in ["loss", "threshold", "accuracy", "precision", "recall", "f1", "f2", "tp", "fp", "tn", "fn"]:
    print(f"{k}: {test_metrics_ae256_fixed[k]}")

=== GatedMLP_AE256 Test ===
loss: 0.5256152389756529
threshold: 0.6952608823776245
accuracy: 0.9727876181120716
precision: 0.6024096385515287
recall: 0.6534365924460144
f1: 0.626886458901327
f2: 0.642551164010585
tp: 1350
fp: 891
tn: 56097
fn: 716
=== GatedMLP_AE256 Test (0.5 threshold) ===
loss: 0.5256152389756529
threshold: 0.5
accuracy: 0.9564297084023172
precision: 0.4269662921336014
recall: 0.7173281703740692
f1: 0.5353079238011325
f2: 0.6314443944963071
tp: 1482
fp: 1989
tn: 54999
fn: 584


In [42]:
print("\n=== Comparison on held-out test ===")
print(f"Baseline V6 F2         : {test_metrics_v2['f2']:.6f}")
print(f"AE16 F2                : {test_metrics_ae16['f2']:.6f}")
print(f"AE256 F2               : {test_metrics_ae256['f2']:.6f}")
print(f"Delta (AE16 vs baseV6) : {test_metrics_ae16['f2'] - test_metrics_v2['f2']:.6f}")
print(f"Delta (AE256 vs baseV6): {test_metrics_ae256['f2'] - test_metrics_v2['f2']:.6f}")

print(f"\nBaseline V6 Precision/Recall: {test_metrics_v2['precision']:.4f} / {test_metrics_v2['recall']:.4f}")
print(f"AE16 Precision/Recall         : {test_metrics_ae16['precision']:.4f} / {test_metrics_ae16['recall']:.4f}")
print(f"AE256 Precision/Recall        : {test_metrics_ae256['precision']:.4f} / {test_metrics_ae256['recall']:.4f}")


=== Comparison on held-out test ===
Baseline V6 F2         : 0.665183
AE16 F2                : 0.674152
AE256 F2               : 0.642551
Delta (AE16 vs baseV6) : 0.008969
Delta (AE256 vs baseV6): -0.022632

Baseline V6 Precision/Recall: 0.6117 / 0.6801
AE16 Precision/Recall         : 0.6214 / 0.6888
AE256 Precision/Recall        : 0.6024 / 0.6534


In [46]:
# checking disagreement rate
def get_probs(model, loader, device):
    model.eval()
    probs = []

    with torch.no_grad():
        for batch in loader:
            if isinstance(batch, (list, tuple)):
                x = batch[0]
            else:
                x = batch

            x = x.to(device)

            logits = model(x)
            p = torch.sigmoid(logits)

            probs.append(p.cpu())

    return torch.cat(probs).squeeze()

thr16 = summary_ae16["best_val_metrics"]["best_threshold"]
thr256 = summary_ae256["best_val_metrics"]["best_threshold"]
thr_base = summary_v2["best_val_metrics"]["best_threshold"]

p_base  = get_probs(model_v2, test_loader, DEVICE) # this is v6 but i am lazy to change the name
p16     = get_probs(model_ae16, test_loader, DEVICE)
p256    = get_probs(model_ae256, test_loader, DEVICE)

pred_base  = (p_base  >= thr_base)
pred16     = (p16     >= thr16)
pred256    = (p256    >= thr256)

all_three = (pred_base & pred16 & pred256)

any_two = (
    (pred_base & pred16 & ~pred256) |
    (pred_base & pred256 & ~pred16) |
    (pred16 & pred256 & ~pred_base)
)

only_one = (
    (pred_base & ~pred16 & ~pred256) |
    (pred16 & ~pred_base & ~pred256) |
    (pred256 & ~pred_base & ~pred16)
)

none = (~pred_base & ~pred16 & ~pred256)

In [48]:
y = test_loader.dataset.y

print("3-way fraud rate :", y[all_three].float().mean().item())
print("2-way fraud rate :", y[any_two].float().mean().item())
print("1-way fraud rate :", y[only_one].float().mean().item())

3-way fraud rate : 0.8240802884101868
2-way fraud rate : 0.2990291118621826
1-way fraud rate : 0.13252094388008118


In [49]:
# agreement rate

print("Agreement breakdown")
print("-------------------")
print("3-way agree :", all_three.float().mean().item())
print("2-way agree :", any_two.float().mean().item())
print("1-way agree :", only_one.float().mean().item())
print("0-way agree :", none.float().mean().item())

Agreement breakdown
-------------------
3-way agree : 0.02531581185758114
2-way agree : 0.00872083194553852
1-way agree : 0.022233888506889343
0-way agree : 0.9437294602394104


In [50]:
# disagreement rate

def disagreement(a, b):
    return (a != b).float().mean().item()

print("Pairwise disagreement rates")
print("---------------------------")
print("Base vs AE16  :", disagreement(pred_base, pred16))
print("Base vs AE256 :", disagreement(pred_base, pred256))
print("AE16 vs AE256 :", disagreement(pred16, pred256))

all_agree = (pred_base == pred16) & (pred16 == pred256)
threeway_disagreement = (~all_agree).float().mean().item()

print("3-way disagreement:", threeway_disagreement)

Pairwise disagreement rates
---------------------------
Base vs AE16  : 0.020675990730524063
Base vs AE256 : 0.02272496372461319
AE16 vs AE256 : 0.018508484587073326
3-way disagreement: 0.030954718589782715


Well my boo boo of accidentally mis labelling the 256 AEmodel might have actually worked out for the best here, there is high disagreement and the agreement metrics look promising for an ensemble:

Pairwise disagreement between models ranged from 1.85% to 2.27%, indicating moderate diversity.
Three-way consensus predictions achieved a fraud rate of 82.4%, compared to 29.9% for two-model agreement and 13.3% for single-model predictions.
This hierarchical agreement structure suggests that model consensus correlates strongly with prediction confidence, motivating a stacked ensemble approach.